In [1]:
# Se importan las librerías usadas para publicar una fuente de serving para BI.

from pathlib import Path
import sqlite3

import pandas as pd
import plotly.express as px

In [2]:
# Se definen el mart de referencia y los productos de serving.

mart_file = Path("../data/sales_mart.db")
submission_directory = Path("../submission")
submission_directory.mkdir(exist_ok=True)
serving_database = submission_directory / "bi_serving.db"

In [3]:
# Se consulta una tabla con el mismo grano que utilizará el dashboard: mes, región y categoría.

serving_query = """
SELECT d.year, d.month, c.region, p.category,
       SUM(f.net_sales) AS net_sales,
       SUM(f.quantity) AS units_sold,
       COUNT(DISTINCT f.order_id) AS orders
FROM fact_sales f
JOIN dim_date d USING(date_key)
JOIN dim_customer c USING(customer_key)
JOIN dim_product p USING(product_key)
GROUP BY d.year, d.month, c.region, p.category
ORDER BY d.year, d.month, c.region, p.category
"""
with sqlite3.connect(mart_file) as connection:
    dashboard_sales = pd.read_sql_query(serving_query, connection)
dashboard_sales.head()

,year,month,region,category,net_sales,units_sold,orders
0,2024,1,Centro,Oficina,1680.0,8,2
1,2024,1,Centro,Servicios,6857.0,11,2
2,2024,1,Centro,Tecnología,3573.0,6,3
3,2024,1,Norte,Oficina,5433.5,24,6
4,2024,1,Norte,Servicios,6323.5,8,4


In [4]:
# Se verifica que el producto de serving tenga un grano único y reconcilie con el mart.

assert not dashboard_sales[["year", "month", "region", "category"]].duplicated().any()
with sqlite3.connect(mart_file) as connection:
    mart_total = connection.execute("SELECT SUM(net_sales) FROM fact_sales").fetchone()[
        0
    ]
assert dashboard_sales["net_sales"].sum() == mart_total
dashboard_sales.shape

(108, 7)

In [5]:
# Se publica la tabla de consumo y una base SQLite lista para un reporte.

dashboard_sales.to_csv(submission_directory / "dashboard_sales.csv", index=False)
with sqlite3.connect(serving_database) as connection:
    dashboard_sales.to_sql(
        "dashboard_sales", connection, index=False, if_exists="replace"
    )

In [6]:
# Se documenta el contrato de consumo para evitar que un dashboard interprete mal el grano publicado.

serving_manifest = pd.DataFrame(
    [
        [
            "dashboard_sales",
            "Una fila por mes, región y categoría",
            "Ventas netas, unidades y órdenes",
            "Dashboard de ventas",
        ],
    ],
    columns=["tabla", "grano", "métricas", "consumidor"],
)
serving_manifest.to_csv(submission_directory / "serving_manifest.csv", index=False)
serving_manifest

,tabla,grano,métricas,consumidor
0,dashboard_sales,"Una fila por mes, región y categoría","Ventas netas, unidades y órdenes",Dashboard de ventas


In [7]:
# ¿Qué región y categoría deben recibir atención en el dashboard de ventas?

priority_view = (
    dashboard_sales.groupby(["region", "category"], as_index=False)["net_sales"]
    .sum()
    .sort_values("net_sales", ascending=False)
)
priority_view.head()

,region,category,net_sales
1,Centro,Servicios,106963.0
8,Sur,Tecnología,99607.5
4,Norte,Servicios,92853.0
7,Sur,Servicios,91112.5
2,Centro,Tecnología,85582.0


In [8]:
# Se visualiza la misma fuente que consumirá el dashboard, sin calcular métricas nuevas en la capa de presentación.

fig = px.bar(
    priority_view,
    x="region",
    y="net_sales",
    color="category",
    barmode="stack",
    title="Ventas netas por región y categoría desde la capa de serving",
    labels={"region": "Región", "net_sales": "Ventas netas", "category": "Categoría"},
)
fig.update_layout(template="plotly_white")
fig.show()